In [14]:
! JuliaSetAnimationGif.f90
! Generates an animated GIF of a morphing Julia set, entirely in Fortran.
! No external tools or languages are used: this program computes the
! fractal AND writes a valid GIF89a file (including a minimal LZW encoder)
! by itself.
!
! Build:   gfortran -O2 JuliaSetAnimationGif.f90 -o JuliaSetAnimationGif
! Run:     ./JuliaSetAnimationGif
! Output:  JuliaSetAnimation.gif

module byte_buffer_mod
    implicit none

    type :: byte_buffer
        integer(kind=1), allocatable :: data(:)
        integer :: length = 0
    end type byte_buffer

contains

    subroutine buf_init(buf, capacity)
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: capacity
        allocate(buf%data(capacity))
        buf%length = 0
    end subroutine buf_init

    subroutine buf_put_byte(buf, value)
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: value   ! 0-255
        buf%length = buf%length + 1
        buf%data(buf%length) = int(mod(value, 256), kind=1)
    end subroutine buf_put_byte

    subroutine buf_put_u16le(buf, value)
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: value
        call buf_put_byte(buf, mod(value, 256))
        call buf_put_byte(buf, value / 256)
    end subroutine buf_put_u16le

    subroutine buf_put_string(buf, s)
        type(byte_buffer), intent(inout) :: buf
        character(len=*), intent(in) :: s
        integer :: i
        do i = 1, len(s)
            call buf_put_byte(buf, iachar(s(i:i)))
        end do
    end subroutine buf_put_string

    subroutine buf_write_to_file(buf, filename)
        type(byte_buffer), intent(in) :: buf
        character(len=*), intent(in) :: filename
        integer :: unit_no

        open(newunit=unit_no, file=filename, access='stream', &
             form='unformatted', status='replace', action='write')
        write(unit_no) buf%data(1:buf%length)
        close(unit_no)
    end subroutine buf_write_to_file

end module byte_buffer_mod


module gif_writer_mod
    use byte_buffer_mod
    implicit none

    ! Simple bit-packer state used while emitting LZW codes.
    type :: bit_packer
        integer :: bit_buf = 0
        integer :: bit_count = 0
        integer(kind=1) :: sub_block(255)
        integer :: sub_len = 0
    end type bit_packer

contains

    subroutine gif_write_header(buf, width, height)
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: width, height
        integer :: i, r, g, b
        real(8) :: hue

        call buf_put_string(buf, 'GIF89a')

        ! Logical Screen Descriptor
        call buf_put_u16le(buf, width)
        call buf_put_u16le(buf, height)
        call buf_put_byte(buf, int(z'F7'))   ! global color table, 256 entries
        call buf_put_byte(buf, 0)            ! background color index
        call buf_put_byte(buf, 0)            ! pixel aspect ratio

        ! Global color table: index 0 is black (points inside the set),
        ! the rest cycle through a rainbow gradient by hue so escaping
        ! points are colored by how quickly they escaped.
        call buf_put_byte(buf, 0)
        call buf_put_byte(buf, 0)
        call buf_put_byte(buf, 0)
        do i = 1, 255
            hue = mod(real(i, 8) * 3.0d0, 256.0d0) / 256.0d0
            call hsv_to_rgb(hue, 0.85d0, 1.0d0, r, g, b)
            call buf_put_byte(buf, r)
            call buf_put_byte(buf, g)
            call buf_put_byte(buf, b)
        end do

        ! Application Extension: NETSCAPE2.0 -> infinite loop
        call buf_put_byte(buf, int(z'21'))
        call buf_put_byte(buf, int(z'FF'))
        call buf_put_byte(buf, 11)
        call buf_put_string(buf, 'NETSCAPE2.0')
        call buf_put_byte(buf, 3)
        call buf_put_byte(buf, 1)
        call buf_put_u16le(buf, 0)   ! loop forever
        call buf_put_byte(buf, 0)
    end subroutine gif_write_header

    ! Converts HSV (all components in [0,1]) to 0-255 RGB integers.
    subroutine hsv_to_rgb(h, s, v, r, g, b)
        real(8), intent(in) :: h, s, v
        integer, intent(out) :: r, g, b
        real(8) :: hh, p, q, t, ff, rr, gg, bb
        integer :: i_sector

        hh = h * 6.0d0
        if (hh >= 6.0d0) hh = 0.0d0
        i_sector = int(hh)
        ff = hh - real(i_sector, 8)

        p = v * (1.0d0 - s)
        q = v * (1.0d0 - s * ff)
        t = v * (1.0d0 - s * (1.0d0 - ff))

        select case (i_sector)
        case (0)
            rr = v; gg = t; bb = p
        case (1)
            rr = q; gg = v; bb = p
        case (2)
            rr = p; gg = v; bb = t
        case (3)
            rr = p; gg = q; bb = v
        case (4)
            rr = t; gg = p; bb = v
        case default
            rr = v; gg = p; bb = q
        end select

        r = min(255, max(0, nint(rr * 255.0d0)))
        g = min(255, max(0, nint(gg * 255.0d0)))
        b = min(255, max(0, nint(bb * 255.0d0)))
    end subroutine hsv_to_rgb

    subroutine gif_write_frame(buf, pixels, width, height, delay_hundredths)
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: width, height, delay_hundredths
        integer, intent(in) :: pixels(width, height)

        ! Graphic Control Extension
        call buf_put_byte(buf, int(z'21'))
        call buf_put_byte(buf, int(z'F9'))
        call buf_put_byte(buf, 4)
        call buf_put_byte(buf, 0)                      ! no disposal specified
        call buf_put_u16le(buf, delay_hundredths)
        call buf_put_byte(buf, 0)                      ! transparent color index
        call buf_put_byte(buf, 0)

        ! Image Descriptor
        call buf_put_byte(buf, int(z'2C'))
        call buf_put_u16le(buf, 0)   ! left
        call buf_put_u16le(buf, 0)   ! top
        call buf_put_u16le(buf, width)
        call buf_put_u16le(buf, height)
        call buf_put_byte(buf, 0)   ! no local color table, no interlace

        ! LZW-compressed image data
        call buf_put_byte(buf, 8)   ! LZW minimum code size
        call lzw_encode_frame(buf, pixels, width, height)
    end subroutine gif_write_frame

    subroutine gif_write_trailer(buf)
        type(byte_buffer), intent(inout) :: buf
        call buf_put_byte(buf, int(z'3B'))
    end subroutine gif_write_trailer

    ! Minimal LZW encoder. It never builds multi-symbol dictionary strings
    ! (each pixel is emitted as its own root code), but it still advances
    ! the "next available code" counter and grows the code size exactly as
    ! a real LZW dictionary would, so that any standard GIF decoder stays
    ! perfectly in sync while decoding. This keeps the encoder simple while
    ! producing a fully valid, if not maximally compressed, GIF stream.
    subroutine lzw_encode_frame(buf, pixels, width, height)
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: width, height
        integer, intent(in) :: pixels(width, height)

        integer, parameter :: min_code_size = 8
        integer, parameter :: clear_code = 256
        integer, parameter :: eoi_code = 257

        type(bit_packer) :: bp
        integer :: code_size, next_code
        integer :: px, py
        logical :: skip_increment

        bp%bit_buf = 0
        bp%bit_count = 0
        bp%sub_len = 0

        code_size = min_code_size + 1
        next_code = eoi_code + 1

        call lzw_emit(bp, buf, clear_code, code_size)

        ! A real decoder adds a new dictionary entry after every emitted
        ! code EXCEPT the very first one following a clear code (there is
        ! no previous string yet to extend). The encoder's simulated
        ! dictionary counter must skip that first increment too, or the
        ! two sides drift out of sync once the code size grows.
        skip_increment = .true.

        do py = 1, height
            do px = 1, width
                call lzw_emit(bp, buf, pixels(px, py), code_size)

                if (skip_increment) then
                    skip_increment = .false.
                else
                    next_code = next_code + 1
                    if (next_code == 4096) then
                        call lzw_emit(bp, buf, clear_code, code_size)
                        next_code = eoi_code + 1
                        code_size = min_code_size + 1
                        skip_increment = .true.
                    else if (next_code == 2**code_size) then
                        code_size = code_size + 1
                    end if
                end if
            end do
        end do

        call lzw_emit(bp, buf, eoi_code, code_size)
        call lzw_flush(bp, buf)
    end subroutine lzw_encode_frame

    subroutine lzw_emit(bp, buf, code, code_size)
        type(bit_packer), intent(inout) :: bp
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: code, code_size

        bp%bit_buf = bp%bit_buf + code * (2 ** bp%bit_count)
        bp%bit_count = bp%bit_count + code_size

        do while (bp%bit_count >= 8)
            call lzw_push_subblock_byte(bp, buf, mod(bp%bit_buf, 256))
            bp%bit_buf = bp%bit_buf / 256
            bp%bit_count = bp%bit_count - 8
        end do
    end subroutine lzw_emit

    subroutine lzw_push_subblock_byte(bp, buf, value)
        type(bit_packer), intent(inout) :: bp
        type(byte_buffer), intent(inout) :: buf
        integer, intent(in) :: value
        integer :: i

        bp%sub_len = bp%sub_len + 1
        bp%sub_block(bp%sub_len) = int(value, kind=1)

        if (bp%sub_len == 255) then
            call buf_put_byte(buf, 255)
            do i = 1, 255
                call buf_put_byte(buf, iand(int(bp%sub_block(i)), 255))
            end do
            bp%sub_len = 0
        end if
    end subroutine lzw_push_subblock_byte

    subroutine lzw_flush(bp, buf)
        type(bit_packer), intent(inout) :: bp
        type(byte_buffer), intent(inout) :: buf
        integer :: i

        if (bp%bit_count > 0) then
            call lzw_push_subblock_byte(bp, buf, mod(bp%bit_buf, 256))
            bp%bit_buf = 0
            bp%bit_count = 0
        end if

        if (bp%sub_len > 0) then
            call buf_put_byte(buf, bp%sub_len)
            do i = 1, bp%sub_len
                call buf_put_byte(buf, iand(int(bp%sub_block(i)), 255))
            end do
            bp%sub_len = 0
        end if

        call buf_put_byte(buf, 0)   ! block terminator
    end subroutine lzw_flush

end module gif_writer_mod


program julia_set_animation_gif
    use byte_buffer_mod
    use gif_writer_mod
    implicit none

    integer, parameter :: width = 600
    integer, parameter :: height = 600
    integer, parameter :: max_iter = 150
    integer, parameter :: n_frames = 60
    integer, parameter :: delay_hundredths = 6   ! 60 ms per frame

    real(8), parameter :: x_min = -1.5d0, x_max = 1.5d0
    real(8), parameter :: y_min = -1.5d0, y_max = 1.5d0
    real(8), parameter :: pi = 3.14159265358979323846d0
    real(8), parameter :: c_radius = 0.7885d0

    type(byte_buffer) :: buf
    integer, allocatable :: pixels(:,:)
    integer :: px, py, iter, frame
    real(8) :: x0, y0, theta, c_re, c_im
    complex(8) :: c, z

    allocate(pixels(width, height))
    call buf_init(buf, 400000000)   ! 400 MB scratch buffer, generous headroom

    call gif_write_header(buf, width, height)

    do frame = 1, n_frames
        theta = 2.0d0 * pi * real(frame - 1, 8) / real(n_frames, 8)
        c_re = c_radius * cos(theta)
        c_im = c_radius * sin(theta)
        c = cmplx(c_re, c_im, kind=8)

        do py = 1, height
            y0 = y_min + (y_max - y_min) * real(py - 1, 8) / real(height - 1, 8)
            do px = 1, width
                x0 = x_min + (x_max - x_min) * real(px - 1, 8) / real(width - 1, 8)
                z = cmplx(x0, y0, kind=8)

                iter = 0
                do while (abs(z) <= 2.0d0 .and. iter < max_iter)
                    z = z*z + c
                    iter = iter + 1
                end do

                if (iter == max_iter) then
                    pixels(px, py) = 0
                else
                    pixels(px, py) = mod(iter * 5, 255) + 1
                end if
            end do
        end do

        call gif_write_frame(buf, pixels, width, height, delay_hundredths)
        print *, 'Frame ', frame, ' / ', n_frames, ' encoded'
    end do

    call gif_write_trailer(buf)
    call buf_write_to_file(buf, 'JuliaSetAnimation.gif')

    deallocate(pixels)
    print *, 'Done: JuliaSetAnimation.gif (', buf%length, ' bytes)'

end program julia_set_animation_gif

 Frame            1  /           60  encoded
 Frame            2  /           60  encoded
 Frame            3  /           60  encoded
 Frame            4  /           60  encoded
 Frame            5  /           60  encoded
 Frame            6  /           60  encoded
 Frame            7  /           60  encoded
 Frame            8  /           60  encoded
 Frame            9  /           60  encoded
 Frame           10  /           60  encoded
 Frame           11  /           60  encoded
 Frame           12  /           60  encoded
 Frame           13  /           60  encoded
 Frame           14  /           60  encoded
 Frame           15  /           60  encoded
 Frame           16  /           60  encoded
 Frame           17  /           60  encoded
 Frame           18  /           60  encoded
 Frame           19  /           60  encoded
 Frame           20  /           60  encoded
 Frame           21  /           60  encoded
 Frame           22  /           60  encoded
 Frame    

![Pendulum animation](JuliaSetAnimation.gif)